# 02. Perceptual-Hash-Based Duplicate Detection

This notebook performs perceptual-hash-based (`pHash`) duplicate and near-duplicate detection on the Paddy Doctor dataset (`data/raw/paddy-disease-classification/train_images/`).

### Motivation & Objectives:
1. **Perceptual Hashing (`pHash`)**:
   Unlike cryptographic hashes (e.g. MD5/SHA256) where even a single altered pixel changes the entire digest, perceptual hashing uses discrete cosine transform (DCT) frequency analysis to construct an invariant 64-bit structural fingerprint. Visually similar or duplicate images—even with slight compression artifacts, resolution shifts, or minor noise—yield very small Hamming distances.
2. **Dataset Audit Goals**:
   - Compute 64-bit pHash for all images across all 10 disease/normal classes.
   - Store results in a structured DataFrame (`file_path`, `class_name`, `phash`).
   - Efficiently search for duplicate and near-duplicate pairs below a Hamming distance threshold of 10.
   - Quantify duplicates within the same class vs. cross-class duplicates (which highlight dataset label noise).
   - Visually inspect the top 15 closest candidate duplicate pairs side-by-side.

## 1. Setup
Import required libraries and resolve the dataset directory identically to `01_data_integrity_and_eda.ipynb`.

In [ ]:
import os
import sys
import time
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import scipy.spatial.distance as dist
import matplotlib.pyplot as plt
import imagehash

# Configure matplotlib inline display
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)

# Resolve dataset directory (supports running from either notebooks/ or project root)
possible_paths = [
    Path("../data/raw/paddy-disease-classification/train_images"),
    Path("data/raw/paddy-disease-classification/train_images"),
]

DATA_DIR = None
for p in possible_paths:
    if p.exists() and p.is_dir():
        DATA_DIR = p.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate 'train_images/'. Please check data/raw/paddy-disease-classification/train_images"
    )

print(f"Dataset directory located at: {DATA_DIR}")
print(f"imagehash version: {imagehash.__version__}")

## 2. Compute pHash for Every Image
Walk through all 10 class subfolders under `train_images/`. For every image:
- Compute the 64-bit perceptual hash using `imagehash.phash(img)`
- Store results in a DataFrame with columns:
  - `file_path`: Full path to the image file
  - `class_name`: Disease category folder name
  - `phash`: Hexadecimal string representation of the perceptual hash

In [ ]:
records = []
class_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])

print(f"Discovering images across {len(class_dirs)} class folders...")
t_start = time.perf_counter()

total_scanned = 0
for class_dir in class_dirs:
    class_name = class_dir.name
    image_files = sorted([f for f in class_dir.iterdir() if f.is_file()])
    for file_path in image_files:
        try:
            with Image.open(file_path) as img:
                h = imagehash.phash(img)
            records.append({
                "file_path": str(file_path),
                "class_name": class_name,
                "phash": str(h)
            })
        except Exception as err:
            print(f"Error hashing {file_path}: {err}")
        total_scanned += 1
        if total_scanned % 2000 == 0:
            print(f"  Processed {total_scanned:,} images...")

t_elapsed = time.perf_counter() - t_start
df_hashes = pd.DataFrame(records)

print(f"\nCompleted pHash computation for {len(df_hashes):,} images in {t_elapsed:.2f} seconds.")
df_hashes.head(10)

## 3. Find Candidate Duplicate Pairs
Compute pairwise Hamming distances between hashes and identify pairs below `HAMMING_THRESHOLD`.

### Pairwise Distance Efficiency Strategy:
The dataset contains $N = 10,407$ images, corresponding to $\frac{N(N-1)}{2} = 54,147,621$ distinct image pairs. A naive nested Python loop ($O(n^2)$) comparing 54M pairs takes tens of minutes.

To compute distances efficiently:
1. Each 64-bit hex hash is unpacked into a 64-element boolean array, forming an $(N, 64)$ boolean matrix.
2. We evaluate pairs using block-chunked pairwise distance computation via SciPy's C-accelerated `scipy.spatial.distance.cdist(chunk, remaining, metric='hamming')`.
3. Multiplying fractional distances by 64 recovers the exact integer Hamming bit difference.
4. Chunking by 1,000 rows avoids allocating huge matrices in memory, running all 54M comparisons in ~2-3 seconds with peak memory bounded under 100 MB.

In [ ]:
# Named threshold for candidate duplicate detection (Hamming distance < HAMMING_THRESHOLD)
HAMMING_THRESHOLD = 10

# Convert hex string hashes into an (N, 64) boolean numpy matrix
print(f"Converting {len(df_hashes):,} hex hashes into 64-bit boolean arrays...")
bool_hashes = np.array([
    imagehash.hex_to_hash(h_str).hash.flatten()
    for h_str in df_hashes["phash"]
], dtype=bool)

num_images = len(df_hashes)
chunk_size = 1000
candidate_pairs = []

print(f"Searching for candidate pairs with Hamming distance < {HAMMING_THRESHOLD}...")
t_start = time.perf_counter()

# Chunked pairwise distance computation:
# Avoids naive O(n²) nested Python loops by evaluating blocks of 1,000 images
# against the remaining dataset using SciPy's vectorized C-optimized hamming distance.
# Multiplied by 64 to convert fractional distance to absolute bit difference.
for start in range(0, num_images, chunk_size):
    end = min(start + chunk_size, num_images)
    chunk = bool_hashes[start:end]
    
    # Compute distances from chunk to all subsequent images (from 'start' to N)
    dists_chunk = dist.cdist(chunk, bool_hashes[start:], metric="hamming") * 64.0
    
    # Identify pairs strictly in the upper triangle (cols > rows) below threshold
    rows, cols = np.where(dists_chunk < HAMMING_THRESHOLD)
    upper_tri_mask = cols > rows
    
    for r, c in zip(rows[upper_tri_mask], cols[upper_tri_mask]):
        idx_a = start + r
        idx_b = start + c
        h_dist = int(round(dists_chunk[r, c]))
        candidate_pairs.append({
            "idx_a": idx_a,
            "idx_b": idx_b,
            "file_a": df_hashes.iloc[idx_a]["file_path"],
            "file_b": df_hashes.iloc[idx_b]["file_path"],
            "class_a": df_hashes.iloc[idx_a]["class_name"],
            "class_b": df_hashes.iloc[idx_b]["class_name"],
            "phash_a": df_hashes.iloc[idx_a]["phash"],
            "phash_b": df_hashes.iloc[idx_b]["phash"],
            "hamming_dist": h_dist,
            "same_class": df_hashes.iloc[idx_a]["class_name"] == df_hashes.iloc[idx_b]["class_name"]
        })

t_elapsed = time.perf_counter() - t_start
print(f"Pairwise search completed in {t_elapsed:.2f} seconds.")
df_duplicates = pd.DataFrame(candidate_pairs)

## 4. Candidate Duplicate Summary
Summarize the detected pairs:
1. Total candidate pairs found (sorted by ascending Hamming distance: most likely duplicates first)
2. Comparison between pairs within the **same class** vs. pairs across **different classes** (cross-class duplicates suggest labeling inconsistencies in the dataset)

In [ ]:
if len(df_duplicates) > 0:
    # Sort pairs by ascending Hamming distance (closest first)
    df_duplicates = df_duplicates.sort_values(by=["hamming_dist", "same_class"], ascending=[True, False]).reset_index(drop=True)
    
    same_class_count = int(df_duplicates["same_class"].sum())
    diff_class_count = len(df_duplicates) - same_class_count
    
    print("=" * 60)
    print("CANDIDATE DUPLICATE DETECTION SUMMARY")
    print("=" * 60)
    print(f"Total candidate duplicate pairs found (Hamming distance < {HAMMING_THRESHOLD}): {len(df_duplicates):,}")
    print(f"  - Within same class:        {same_class_count:,} pairs ({same_class_count / len(df_duplicates) * 100:.1f}%)")
    print(f"  - Across different classes: {diff_class_count:,} pairs ({diff_class_count / len(df_duplicates) * 100:.1f}%)")
    
    print("\nBreakdown by Hamming Distance:")
    dist_counts = df_duplicates["hamming_dist"].value_counts().sort_index()
    for d, count in dist_counts.items():
        same_d = ((df_duplicates["hamming_dist"] == d) & (df_duplicates["same_class"])).sum()
        diff_d = ((df_duplicates["hamming_dist"] == d) & (~df_duplicates["same_class"])).sum()
        print(f"  - Distance {d:>2}: {count:>5} pairs (Same class: {same_d:>4}, Cross class: {diff_d:>4})")
    
    if diff_class_count > 0:
        print("\n[NOTE] Cross-class duplicate candidates detected. These warrant review for potential label noise.")
    
    print("\nTop 10 Candidate Pairs Preview:")
    display_cols = ["hamming_dist", "same_class", "class_a", "class_b", "file_a", "file_b"]
    display(df_duplicates[display_cols].head(10))
else:
    print(f"No candidate pairs found with Hamming distance < {HAMMING_THRESHOLD}.")

## 5. Visualize Top Candidate Duplicate Pairs
Display the 15 closest candidate pairs (smallest Hamming distance) side-by-side with their file paths, class names, and Hamming distance as the title.

In [ ]:
TOP_N = 15

if len(df_duplicates) > 0:
    num_to_display = min(TOP_N, len(df_duplicates))
    print(f"Displaying top {num_to_display} closest candidate duplicate pairs:\n")
    
    for rank in range(num_to_display):
        row = df_duplicates.iloc[rank]
        
        fig, axes = plt.subplots(1, 2, figsize=(11, 5))
        
        # Load and render Image A
        img_a = Image.open(row["file_a"])
        axes[0].imshow(img_a)
        fname_a = Path(row["file_a"]).name
        axes[0].set_title(f"Image A: {fname_a}\nClass: {row['class_a']}\n{row['file_a']}", fontsize=8)
        axes[0].axis("off")
        
        # Load and render Image B
        img_b = Image.open(row["file_b"])
        axes[1].imshow(img_b)
        fname_b = Path(row["file_b"]).name
        axes[1].set_title(f"Image B: {fname_b}\nClass: {row['class_b']}\n{row['file_b']}", fontsize=8)
        axes[1].axis("off")
        
        class_status = "SAME CLASS" if row["same_class"] else "CROSS CLASS (LABEL CONFLICT)"
        title_color = "red" if not row["same_class"] else "navy"
        fig.suptitle(
            f"Rank #{rank + 1} | Hamming Distance: {row['hamming_dist']} ({class_status})",
            fontsize=12,
            fontweight="bold",
            color=title_color,
            y=1.02
        )
        
        plt.tight_layout()
        plt.show()
else:
    print("No pairs available for visualization.")

## 6. Full Distance-0 Tier Breakdown
Isolate all 229 pairs at Hamming distance exactly 0 (identical perceptual hashes) into a dedicated DataFrame `df_dist0`.

### What is Being Checked & Why:
- **Exact Perceptual Duplicates**: A Hamming distance of 0 means all 64 bits of the perceptual hash match. This indicates identical or near-identical image structure (such as identical photos, re-encoded files, or slight rescales).
- **Class-by-Class Distribution**: We evaluate how many distance-0 pairs fall into each of the 10 disease categories to identify which classes have the highest duplication rates.
- **Cross-Class Pairs (Potential Label Noise)**: While same-class duplicates represent redundancy, pairs across different classes that share identical pHash values point to potential labeling errors or ambiguous symptoms across disease classes. These are explicitly extracted and listed with their file paths and labels for direct inspection.

In [ ]:
# Filter all candidate pairs at Hamming distance exactly 0
df_dist0 = df_duplicates[df_duplicates["hamming_dist"] == 0].copy().reset_index(drop=True)

same_class_count_0 = int(df_dist0["same_class"].sum())
cross_class_count_0 = len(df_dist0) - same_class_count_0

print("=" * 60)
print("HAMMING DISTANCE 0 TIER ANALYSIS")
print("=" * 60)
print(f"Total pairs at Hamming distance == 0: {len(df_dist0):,}")
print(f"  - Same-class pairs:  {same_class_count_0:,} ({same_class_count_0 / len(df_dist0) * 100:.1f}%)")
print(f"  - Cross-class pairs: {cross_class_count_0:,} ({cross_class_count_0 / len(df_dist0) * 100:.1f}%)")

# Full class-by-class breakdown
all_classes = sorted(list(set(df_hashes["class_name"])))
breakdown_records = []

for c in all_classes:
    same_c = int(((df_dist0["same_class"]) & (df_dist0["class_a"] == c)).sum())
    cross_c = int(((~df_dist0["same_class"]) & ((df_dist0["class_a"] == c) | (df_dist0["class_b"] == c))).sum())
    breakdown_records.append({
        "Class": c,
        "Same-Class Pairs": same_c,
        "Cross-Class Pairs": cross_c,
        "Total Pairs Involved": same_c + cross_c
    })

df_dist0_class_breakdown = pd.DataFrame(breakdown_records)
print("\nClass-by-Class Breakdown (Hamming Distance = 0):")
display(df_dist0_class_breakdown)

# Explicitly list the cross-class pairs at distance 0
df_dist0_cross = df_dist0[~df_dist0["same_class"]].reset_index(drop=True)
print(f"\nExplicit Listing of Cross-Class Pairs at Distance 0 ({len(df_dist0_cross)} pairs):")
display_cols = ["class_a", "class_b", "file_a", "file_b", "phash_a", "phash_b", "hamming_dist"]
display(df_dist0_cross[display_cols])

for i, row in df_dist0_cross.iterrows():
    print(f"\n[Cross-Class Pair #{i + 1}]")
    print(f"  Class A: {row['class_a']}")
    print(f"  File A:  {row['file_a']}")
    print(f"  Class B: {row['class_b']}")
    print(f"  File B:  {row['file_b']}")
    print(f"  pHash:   {row['phash_a']}")


## 7. High-Distance Spot Check (Hamming Distance 8 Tier)
Randomly sample 10 pairs from the distance 8 tier (3,401 pairs) using a fixed random seed for reproducible visual inspection.

### What is Being Checked & Why:
- **Assessing Semantic Validity at Higher Distances**: The distance 8 tier represents the largest candidate duplicate cluster (3,401 pairs) under `HAMMING_THRESHOLD = 10`. At 8 bit differences out of 64, these pairs may either represent genuine near-duplicates (e.g. slight camera angle/zoom variations of the same plant) or structural false positives (distinct plants in similar field environments sharing overall lighting, background soil, or broad canopy layout).
- **Reproducible Random Sampling**: A fixed random seed (`random_state=42`) is used to sample 10 pairs from this tier without selection bias.
- **Side-by-Side Visualization**: Each sampled pair is rendered side-by-side using the same visual format as Section 5, showing file paths, class labels, and Hamming distance to facilitate visual inspection.

In [ ]:
RANDOM_SEED = 42
NUM_SAMPLES = 10

# Filter for the distance 8 tier
df_dist8 = df_duplicates[df_duplicates["hamming_dist"] == 8].copy().reset_index(drop=True)
print(f"Total candidate duplicate pairs at Hamming distance == 8: {len(df_dist8):,}")

# Sample 10 pairs reproducibly
df_dist8_sample = df_dist8.sample(n=NUM_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Displaying {len(df_dist8_sample)} randomly sampled pairs from distance 8 tier (seed={RANDOM_SEED}):\n")

for i, row in df_dist8_sample.iterrows():
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    
    # Load and render Image A
    img_a = Image.open(row["file_a"])
    axes[0].imshow(img_a)
    fname_a = Path(row["file_a"]).name
    axes[0].set_title(f"Image A: {fname_a}\nClass: {row['class_a']}\n{row['file_a']}", fontsize=8)
    axes[0].axis("off")
    
    # Load and render Image B
    img_b = Image.open(row["file_b"])
    axes[1].imshow(img_b)
    fname_b = Path(row["file_b"]).name
    axes[1].set_title(f"Image B: {fname_b}\nClass: {row['class_b']}\n{row['file_b']}", fontsize=8)
    axes[1].axis("off")
    
    class_status = "SAME CLASS" if row["same_class"] else "CROSS CLASS (LABEL CONFLICT)"
    title_color = "red" if not row["same_class"] else "navy"
    fig.suptitle(
        f"Distance 8 Spot Check #{i + 1} | Hamming Distance: {row['hamming_dist']} ({class_status})",
        fontsize=12,
        fontweight="bold",
        color=title_color,
        y=1.02
    )
    
    plt.tight_layout()
    plt.show()


## 8. Observations and Next Steps (pHash Summary)

- pHash (64-bit, imagehash 4.3.2) was computed for all 10,407 images in 51.45 s; the chunked pairwise Hamming search over about 54M pairs took 5.10 s.
- 9,106 candidate pairs fell under distance 10. Breakdown by Hamming distance: 0: 229, 2: 822, 4: 1,910, 6: 2,744, 8: 3,401. At distance 0: 227 same-class pairs and 2 cross-class pairs.
- Visual checks: distance-0 pairs checked in bacterial_leaf_blight, bacterial_leaf_streak, hispa and blast were identical images. A seed-42 sample of 10 pairs at distance 8 were different photos with similar lighting and background, so pHash over-flags above distance 0. The actionable threshold is exactly 0.
- Cross-class pairs: blast/105300.jpg with downy_mildew/100821.jpg, and hispa/107223.jpg with tungro/109617.jpg. Each pair is the same image carrying two different labels (a labelling error). Both copies of each pair are removed.
- Early hypothesis corrected: the first 15 pairs came from the two smallest classes, but the full distance-0 tier shows duplicates concentrated in hispa (47 pairs) and blast (35), with bacterial_panicle_blight at 1. The preview pattern was an ordering artefact.
- Clustering: 10,203 connected components; 184 multi-image clusters (169 pairs, 10 triplets, 5 quadruplets) covering 388 images. Removal: 206 images (202 duplicates, 4 conflict images), 1.98% of the data; 10,201 kept. Removed per class: hispa 38, blast 33, dead_heart 29, brown_spot 28, normal 22, tungro 18, bacterial_leaf_blight 13, bacterial_leaf_streak 13, downy_mildew 11, bacterial_panicle_blight 1.
- Rule: within a same-class cluster, keep the lowest numeric filename (reproducible, not quality-based). data/raw is untouched; decisions are recorded in data/processed/dedup_manifest.csv. The cleanup logic is being moved to src/data_prep/dedup.py with tests.


## 9. Connected Component Clustering & Deduplication Manifest

Group all 229 distance-0 candidate duplicate pairs (227 same-class pairs and 2 cross-class pairs) into connected components using graph theory via `scipy.sparse.csgraph.connected_components`.

### Deduplication Strategy & Methodology:
1. **Graph Connected Components**:
   - Each image in the dataset is modeled as a graph node ($N = 10,407$).
   - Each distance-0 duplicate pair defines an undirected edge between two nodes.
   - Running `connected_components(csgraph=adj, directed=False)` unifies all duplicate groups (including triplets and quadruplets) into coherent clusters, ensuring multi-duplicate transitive relationships are handled rigorously rather than treating pairs as isolated.

2. **Deterministic Retention Policy (Same-Class Clusters)**:
   - For every cluster consisting entirely of images within the same disease class, keep exactly one representative image.
   - **Deterministic Selection Rule**: The image with the lowest numeric filename stem (e.g., `100123.jpg` before `100456.jpg`) is retained as the representative image to guarantee 100% reproducible execution across runs.
   - All other images in the cluster are marked `status = "remove"` with `reason = "duplicate"`.

3. **Cross-Class Label Conflict Policy**:
   - For the 2 known cross-class clusters (`blast/105300.jpg` + `downy_mildew/100821.jpg`, and `hispa/107223.jpg` + `tungro/109617.jpg`), mark **both** images in each cluster as `status = "remove"` with `reason = "cross_class_label_conflict"`. Neither image is retained to prevent contradictory labels from contaminating model training.

4. **Complete Dataset Manifest Generation**:
   - Export `data/processed/dedup_manifest.csv` covering all 10,407 images with columns: `file_path`, `class_name`, `status`, `reason`, `cluster_id`.
   - The manifest is a complete, self-contained record of the cleaned dataset (kept rows included).
   - Raw images in `data/raw/` remain strictly untouched.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

# 1. Build adjacency graph for distance-0 pairs across all dataset images
num_images = len(df_hashes)
file2idx = {p: i for i, p in enumerate(df_hashes["file_path"])}

row_ind = [file2idx[p] for p in df_dist0["file_a"]]
col_ind = [file2idx[p] for p in df_dist0["file_b"]]
data = np.ones(len(df_dist0), dtype=int)

adj_matrix = csr_matrix((data, (row_ind, col_ind)), shape=(num_images, num_images))
adj_matrix = adj_matrix + adj_matrix.T

# Compute connected components
n_components, comp_labels = connected_components(csgraph=adj_matrix, directed=False, return_labels=True)
df_hashes["comp_id"] = comp_labels
cluster_sizes = df_hashes.groupby("comp_id").size()

dup_cluster_ids = cluster_sizes[cluster_sizes > 1].index.tolist()

print("=" * 70)
print("DISTANCE-0 CONNECTED COMPONENT CLUSTERING")
print("=" * 70)
print(f"Total images in dataset:                        {num_images:,}")
print(f"Total connected components:                     {n_components:,}")
print(f"Total multi-image duplicate clusters (size > 1): {len(dup_cluster_ids):,}")

size_dist = cluster_sizes[cluster_sizes > 1].value_counts().sort_index()
for sz, count in size_dist.items():
    print(f"  - Clusters of size {sz}: {count:>3} clusters ({sz * count:>3} images total)")

# 2. Assign keep/remove status, reasons, and cluster IDs
manifest_records = []
cluster_counter = 1
same_class_cluster_count = 0
cross_class_cluster_count = 0

for cid in dup_cluster_ids:
    sub_df = df_hashes[df_hashes["comp_id"] == cid]
    classes_in_cluster = sub_df["class_name"].unique()
    
    if len(classes_in_cluster) > 1:
        # Cross-class conflict: mark both images in cluster as remove
        cross_class_cluster_count += 1
        for idx, row in sub_df.iterrows():
            manifest_records.append({
                "orig_idx": idx,
                "file_path": row["file_path"],
                "class_name": row["class_name"],
                "status": "remove",
                "reason": "cross_class_label_conflict",
                "cluster_id": cluster_counter
            })
    else:
        # Same-class cluster: keep exactly one representative image deterministically.
        # Deterministic selection rule: choose the image with the lowest numeric filename stem.
        same_class_cluster_count += 1
        sub_df_sorted = sub_df.sort_values(
            by="file_path",
            key=lambda s: s.map(
                lambda p: int(re.search(r'\d+', Path(p).stem).group())
                if re.search(r'\d+', Path(p).stem) else Path(p).name
            )
        )
        kept_idx = sub_df_sorted.index[0]
        for idx, row in sub_df_sorted.iterrows():
            if idx == kept_idx:
                manifest_records.append({
                    "orig_idx": idx,
                    "file_path": row["file_path"],
                    "class_name": row["class_name"],
                    "status": "keep",
                    "reason": "",  # Blank for kept images
                    "cluster_id": cluster_counter
                })
            else:
                manifest_records.append({
                    "orig_idx": idx,
                    "file_path": row["file_path"],
                    "class_name": row["class_name"],
                    "status": "remove",
                    "reason": "duplicate",
                    "cluster_id": cluster_counter
                })
    cluster_counter += 1

# 3. Add all unclustered singleton images (unique images with no distance-0 duplicates)
singleton_cids = cluster_sizes[cluster_sizes == 1].index.tolist()
for cid in singleton_cids:
    idx = df_hashes[df_hashes["comp_id"] == cid].index[0]
    row = df_hashes.loc[idx]
    manifest_records.append({
        "orig_idx": idx,
        "file_path": row["file_path"],
        "class_name": row["class_name"],
        "status": "keep",
        "reason": "",  # Blank for kept images
        "cluster_id": pd.NA  # Blank for unclustered singletons
    })

# Sort back to original dataset ordering
df_manifest = pd.DataFrame(manifest_records).sort_values("orig_idx").drop(columns=["orig_idx"]).reset_index(drop=True)

# 4. Print summary of removals
total_removed = (df_manifest["status"] == "remove").sum()
total_kept = (df_manifest["status"] == "keep").sum()
rem_dup = (df_manifest["reason"] == "duplicate").sum()
rem_conflict = (df_manifest["reason"] == "cross_class_label_conflict").sum()

print("\n" + "=" * 70)
print("DEDUPLICATION REMOVAL SUMMARY")
print("=" * 70)
print(f"Total images in dataset:              {len(df_manifest):,} (100.00%)")
print(f"Total images KEPT:                    {total_kept:,} ({total_kept / len(df_manifest) * 100:.2f}%)")
print(f"Total images MARKED FOR REMOVAL:      {total_removed:,} ({total_removed / len(df_manifest) * 100:.2f}%)")
print(f"  - Reason 'duplicate':                 {rem_dup:>5,} images (from {same_class_cluster_count} same-class clusters)")
print(f"  - Reason 'cross_class_label_conflict':{rem_conflict:>5,} images (from {cross_class_cluster_count} cross-class clusters)")

# 5. Per-class removal breakdown and impact evaluation
per_class_orig = df_hashes["class_name"].value_counts()
per_class_rem = df_manifest[df_manifest["status"] == "remove"]["class_name"].value_counts().reindex(per_class_orig.index, fill_value=0)
per_class_kept = per_class_orig - per_class_rem
per_class_pct = (per_class_rem / per_class_orig * 100).round(2)

df_per_class_impact = pd.DataFrame({
    "Original Images": per_class_orig,
    "Images Removed": per_class_rem,
    "Remaining Images": per_class_kept,
    "Removal %": per_class_pct
})

print("\n" + "=" * 70)
print("PER-CLASS REMOVAL BREAKDOWN & CLASS SIZE IMPACT")
print("=" * 70)
display(df_per_class_impact)

### Export Cleaned Dataset Manifest (`dedup_manifest.csv`)

Write the complete manifest covering all 10,407 images to `data/processed/dedup_manifest.csv`.
- Raw image files in `data/raw/` remain untouched.
- Downstream training and cross-validation pipelines can filter `status == 'keep'` to load the deduplicated dataset, or group by `cluster_id` where applicable.

In [ ]:
# Create data/processed/ directory if it doesn't exist
processed_dir = Path("../data/processed") if Path("../data").exists() else Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
manifest_path = processed_dir / "dedup_manifest.csv"

# Normalize file paths to relative POSIX paths from project root for portability
def normalize_path(p):
    p_obj = Path(p)
    if "data" in p_obj.parts:
        idx = p_obj.parts.index("data")
        return "/".join(p_obj.parts[idx:])
    return str(p_obj).replace("\\", "/")

df_manifest_export = df_manifest.copy()
df_manifest_export["file_path"] = df_manifest_export["file_path"].apply(normalize_path)
df_manifest_export["cluster_id"] = df_manifest_export["cluster_id"].astype("Int64")

# Write manifest to CSV
df_manifest_export.to_csv(manifest_path, index=False)
print(f"Manifest file successfully written to: {manifest_path.resolve()}")
print(f"Manifest shape: {df_manifest_export.shape} (Total rows: {len(df_manifest_export):,})")

print("\nFirst 10 rows of manifest:")
display(df_manifest_export.head(10))

print("\nFirst 5 removed duplicate rows (status == 'remove', reason == 'duplicate'):")
display(df_manifest_export[df_manifest_export["reason"] == "duplicate"].head(5))

print("\nAll cross-class label conflict rows (reason == 'cross_class_label_conflict'):")
display(df_manifest_export[df_manifest_export["reason"] == "cross_class_label_conflict"])

## 10. Targeted Visual Spot Check: Distance-0 Pairs in Top Duplicated Classes (`hispa` & `blast`)

This is a targeted visual spot check of the two classes with the largest volume of distance-0 duplicate pairs (`hispa` and `blast`), which had not been directly visually verified in earlier sections.

We filter `df_dist0` for same-class pairs in each category, sample 4 pairs from `hispa` and 4 pairs from `blast` reproducibly (`random_state=42`), and render them side-by-side using the standard spot check format.

In [ ]:
RANDOM_SEED = 42

# Filter for same-class distance-0 pairs in hispa and blast
df_dist0_hispa = (
    df_dist0[(df_dist0["same_class"]) & (df_dist0["class_a"] == "hispa")]
    .sample(n=4, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

df_dist0_blast = (
    df_dist0[(df_dist0["same_class"]) & (df_dist0["class_a"] == "blast")]
    .sample(n=4, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

# Combine into targeted evaluation set (4 hispa + 4 blast = 8 pairs)
df_targeted = pd.concat([df_dist0_hispa, df_dist0_blast], ignore_index=True)

print(f"Displaying {len(df_targeted)} targeted distance-0 pairs (4 hispa, 4 blast, seed={RANDOM_SEED}):\n")

for i, row in df_targeted.iterrows():
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    
    # Load and render Image A
    img_a = Image.open(row["file_a"])
    axes[0].imshow(img_a)
    fname_a = Path(row["file_a"]).name
    axes[0].set_title(f"Image A: {fname_a}\nClass: {row['class_a']}\n{row['file_a']}", fontsize=8)
    axes[0].axis("off")
    
    # Load and render Image B
    img_b = Image.open(row["file_b"])
    axes[1].imshow(img_b)
    fname_b = Path(row["file_b"]).name
    axes[1].set_title(f"Image B: {fname_b}\nClass: {row['class_b']}\n{row['file_b']}", fontsize=8)
    axes[1].axis("off")
    
    class_status = "SAME CLASS" if row["same_class"] else "CROSS CLASS (LABEL CONFLICT)"
    title_color = "navy" if row["same_class"] else "red"
    fig.suptitle(
        f"Targeted Spot Check #{i + 1} ({row['class_a']}) | Hamming Distance: {row['hamming_dist']} ({class_status})",
        fontsize=12,
        fontweight="bold",
        color=title_color,
        y=1.02
    )
    
    plt.tight_layout()
    plt.show()
